In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt

from src.rl.env_autoscale import AutoscaleEnv
from src.rl.agent_ppo import PPOAgent


In [ ]:
agent = PPOAgent(state_dim=4, action_dim=3)
agent.policy.load_state_dict(torch.load("models/rl_policy.pt"))
agent.value.load_state_dict(torch.load("models/rl_value.pt"))


In [ ]:
def policy_over_cpu():
    cpu_values = np.linspace(0.1, 1.0, 20)
    probs_scale_down = []
    probs_hold = []
    probs_scale_up = []

    for cpu in cpu_values:
        state = np.array([cpu, 0.2, 0.2, 0.3], dtype=np.float32)
        state_t = torch.tensor(state, dtype=torch.float32).unsqueeze(0)
        logits = agent.policy(state_t)
        probs = torch.softmax(logits, dim=-1).detach().numpy()[0]

        probs_scale_down.append(probs[0])
        probs_hold.append(probs[1])
        probs_scale_up.append(probs[2])

    plt.figure(figsize=(10,5))
    plt.plot(cpu_values, probs_scale_down, label="scale_down")
    plt.plot(cpu_values, probs_hold, label="hold")
    plt.plot(cpu_values, probs_scale_up, label="scale_up")
    plt.xlabel("CPU usage")
    plt.ylabel("Action probability")
    plt.title("Policy behavior vs CPU usage")
    plt.legend()
    plt.grid(True)
    plt.show()

policy_over_cpu()


In [ ]:
env = AutoscaleEnv()
state = env.reset()

for i in range(15):
    state_t = torch.tensor(state, dtype=torch.float32).unsqueeze(0)
    logits = agent.policy(state_t)
    probs = torch.softmax(logits, dim=-1)
    action = torch.argmax(probs, dim=-1).item()

    next_state, reward, done, _ = env.step(action)

    print(f"Step {i}: state={state}, action={action}, reward={reward:.4f}, replicas={env.replicas}")

    state = next_state
    if done:
        break
